(cap-metricas-centralidade)=
# Métricas Estruturais e Centralidade em Redes Espaciais

:::{admonition} Objetivos de Aprendizagem
:class: important

Ao final deste capítulo, você será capaz de:

1. **Calcular e interpretar** as estatísticas básicas de uma rede viária (topológicas, geométricas e combinadas), incluindo densidades por área e circuidade média.
2. **Aplicar** os índices clássicos de conectividade da geografia dos transportes ($\beta$, $\gamma$, $\alpha$ e número ciclomático) e reconhecer as hipóteses de planaridade que os fundamentam.
3. **Formalizar** a densidade, a assortatividade de grau e as eficiências global e local, reconhecendo suas limitações em grafos espaciais.
4. **Definir matematicamente** as centralidades de grau, de proximidade e de intermediação, nas versões direcionadas e não direcionadas, e **implementá-las** corretamente com a NetworkX.
5. **Distinguir** as proximidades de entrada e de saída em multidígrafos, evitando uma inversão comum na literatura aplicada.
6. **Quantificar** a orientação das vias por meio da entropia de orientação e do índice de ordem $\varphi$.
:::

O {ref}`cap-osmnx` mostrou como transformar dados do OpenStreetMap em um multidígrafo analisável. Este capítulo responde à pergunta seguinte: **como descrever quantitativamente essa rede?** Começamos por estatísticas *descritivas* da forma e do tamanho da rede, passamos a indicadores de *conectividade* e chegamos às medidas de *centralidade*, que atribuem a cada interseção (ou segmento) um valor de importância estrutural. Encerramos com a análise da *orientação* das vias, uma assinatura morfológica do tecido urbano.

Um fio condutor atravessa o capítulo: **métricas desenvolvidas para redes abstratas precisam ser reinterpretadas em redes espaciais**. A restrição de grau, a quase-planaridade e o papel da distância física, discutidos no {ref}`cap-fundamentos-grafos`, alteram o significado e a faixa de variação de muitas delas {cite:p}`barthelemy2011,porta2006`.

## Preparação dos dados

Continuamos com o bairro da Liberdade, em São Paulo, e a rede para automóveis (`network_type="drive"`), cujo sentido de circulação é relevante para a análise de centralidade.

In [ ]:
# Descomente a linha abaixo se estiver executando no Google Colab
# %pip install -q osmnx mapclassify

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

ox.settings.use_cache = True

bairro = "Liberdade, São Paulo, Brasil"
G = ox.graph_from_place(bairro, network_type="drive")
print(f"Multidígrafo: n = {len(G)} nós, m = {G.number_of_edges()} arestas")

In [ ]:
fig, ax = ox.plot_graph(G, node_size=8, node_color="tab:red", edge_color="gray",
                        edge_linewidth=0.8, bgcolor="white")

:::{warning}
Os valores numéricos deste capítulo dependem do estado do OpenStreetMap no momento da execução e da versão da OSMnx. As **interpretações** apresentadas no texto referem-se a padrões qualitativos robustos; ao reproduzir a análise, confira os números obtidos e ajuste a leitura se necessário.
:::

## Estatísticas básicas da rede

As **estatísticas básicas** caracterizam o tamanho, a forma e a compacidade da rede. Seguindo {cite:t}`boeing2017`, agrupamo-las em três famílias:

- **topológicas**, que dependem apenas da estrutura de conexões;
- **geométricas**, que dependem de comprimentos;
- **combinadas**, que relacionam estrutura, geometria e área.

As densidades por área exigem a área de estudo em unidades métricas. Projetamos o polígono do bairro para UTM e extraímos sua área como um **número real**, em m²:

In [ ]:
poligono = ox.geocode_to_gdf(bairro)
area_m2 = float(ox.projection.project_gdf(poligono).area.iloc[0])
print(f"Área de estudo: {area_m2 / 1e6:.3f} km²")

:::{note}
O argumento `area` de `ox.basic_stats` deve ser um valor escalar. Passar uma `GeoSeries` (o resultado direto de `.area`) propaga um objeto pandas para todas as densidades calculadas, o que dificulta sua leitura e pode gerar erros em versões recentes da biblioteca.
:::

In [ ]:
stats = ox.basic_stats(G, area=area_m2)

# Desempacota os dicionários de contagem e proporção de ruas por nó
for k, v in stats.pop("streets_per_node_counts").items():
    stats[f"nos_com_{k}_ruas_contagem"] = v
for k, v in stats.pop("streets_per_node_proportions").items():
    stats[f"nos_com_{k}_ruas_proporcao"] = v

estatisticas = pd.Series(stats, name="valor").round(4).to_frame()
estatisticas

A {numref}`tab-basic-stats` formaliza as principais estatísticas. Nas fórmulas, $G_u$ denota o grafo **não direcionado** obtido com `ox.convert.to_undirected`, no qual cada rua de mão dupla é contada uma única vez; $\ell_e$ é o comprimento da aresta $e$; $s(v)$ é o `street_count` do nó $v$; e $A$ é a área de estudo em km².

```{list-table} Estatísticas básicas calculadas pela função basic_stats da OSMnx.
:header-rows: 1
:name: tab-basic-stats
:widths: 22 38 40

* - Chave
  - Definição
  - Interpretação
* - `n`, `m`
  - $\lvert V \rvert$, $\lvert E \rvert$ no multidígrafo
  - Tamanho da rede (arcos contam separadamente por sentido)
* - `k_avg`
  - $\bar{k} = 2m/n$
  - Grau médio no multidígrafo
* - `streets_per_node_avg`
  - $\frac{1}{n}\sum_v s(v)$
  - Número médio de ruas físicas por nó
* - `intersection_count`
  - $\lvert\{v : s(v) > 1\}\rvert$
  - Nós que não são becos sem saída
* - `edge_length_total`
  - $\sum_{e \in E} \ell_e$
  - Extensão total, contando ambos os sentidos
* - `street_length_total`
  - $\sum_{e \in E(G_u)} \ell_e$
  - Extensão física das ruas
* - `street_segment_count`
  - $\lvert E(G_u) \rvert$
  - Número de segmentos físicos
* - `circuity_avg`
  - $\dfrac{\sum_{e \in E(G_u)} \ell_e}{\sum_{(u,v) \in E(G_u)} d_{gc}(u,v)}$
  - Sinuosidade média dos segmentos ($\geq 1$)
* - `node_density_km`, `intersection_density_km`
  - $n/A$, $\text{intersection\_count}/A$
  - Nós e interseções por km²
* - `street_density_km`
  - $\text{street\_length\_total}/A$
  - Metros de rua por km²
* - `self_loop_proportion`
  - $\lvert\{(u,v): u=v\}\rvert / m$
  - Proporção de autolaços
```

### Leitura das estatísticas topológicas

Em redes viárias da OSMnx, $\bar{k}$ (`k_avg`) é tipicamente maior que `streets_per_node_avg`, porque cada rua de mão dupla contribui com **dois** arcos para o grau, mas conta como **uma** rua para `street_count` (ver o {ref}`cap-osmnx`). Em áreas centrais com muitas vias de mão única, a razão entre as duas medidas se aproxima de 1; em bairros residenciais de mão dupla, aproxima-se de 2.

A distribuição de `street_count`, obtida nas chaves `nos_com_k_ruas_*`, é uma assinatura morfológica. Predominância de nós com 3 ruas indica tecido com muitas junções em "T", típico de loteamentos orgânicos ou de quadras irregulares. Predominância de nós com 4 ruas indica malha em grade. A proporção de nós com uma única rua (becos sem saída) é um indicador inverso de permeabilidade da malha.

In [ ]:
contagem = pd.Series({k: v for k, v in stats.items() if k.endswith("_ruas_proporcao")})
contagem.index = [int(i.split("_")[2]) for i in contagem.index]
ax = contagem.sort_index().plot.bar(color="tab:blue", figsize=(6, 3.5))
ax.set_xlabel("Ruas que convergem no nó (street_count)")
ax.set_ylabel("Proporção de nós")
ax.set_title("Distribuição do número de ruas por nó")
plt.show()

### Autolaços

Autolaços, isto é, arestas com $u = v$, costumam representar vias em laço, como retornos e pequenas rotatórias mapeadas como um único *way* fechado. A proporção é em geral ínfima, mas localizá-los ajuda a detectar particularidades do mapeamento:

In [ ]:
autolacos = [(u, v, k) for u, v, k in G.edges(keys=True) if u == v]
print("Autolaços encontrados:", len(autolacos))

cores = ["red" if u == v else "lightgray" for u, v, k in G.edges(keys=True)]
larguras = [3 if u == v else 0.6 for u, v, k in G.edges(keys=True)]
fig, ax = ox.plot_graph(G, edge_color=cores, edge_linewidth=larguras, node_size=0, bgcolor="white")

### Leitura das estatísticas geométricas e combinadas

A diferença entre `edge_length_total` e `street_length_total` decorre da duplicação de arcos em vias de mão dupla, e sua razão estima a fração de extensão viária com circulação nos dois sentidos. A **circuidade média** mede quanto os segmentos se desviam da linha reta: valores próximos de 1,0 são comuns em áreas urbanas consolidadas, enquanto valores maiores indicam vias sinuosas, que acompanham o relevo, por exemplo. Note que essa circuidade é calculada **por segmento** e é diferente da circuidade **entre pares origem–destino** definida no {ref}`cap-fundamentos-grafos`, que incorpora os desvios impostos pela topologia da rede.

As **densidades** por km² permitem comparar áreas de tamanhos distintos. A densidade de interseções, em particular, é um dos indicadores mais usados em estudos de caminhabilidade: malhas finas, com muitas interseções por km², oferecem mais alternativas de percurso e quarteirões menores.

## Índices clássicos de conectividade

A geografia dos transportes desenvolveu, na década de 1960, índices simples para comparar a conectividade de redes {cite:p}`kansky1963`. Eles são calculados sobre o grafo não direcionado e **simples** que representa as ligações físicas, com $n$ vértices, $m$ arestas e $p$ componentes conexas.

- **Índice beta**, a razão entre arestas e vértices:

  $$\beta = \frac{m}{n}.$$ (eq-beta)

- **Número ciclomático**, o número de circuitos independentes da rede:

  $$\mu = m - n + p.$$ (eq-ciclomatico)

- **Índice gama**, a razão entre as arestas existentes e o máximo possível em um grafo **planar** simples, $3(n-2)$:

  $$\gamma = \frac{m}{3(n - 2)}.$$ (eq-gama)

- **Índice alfa**, a razão entre os circuitos existentes e o máximo possível em um grafo planar, $2n - 5$:

  $$\alpha = \frac{m - n + p}{2n - 5}.$$ (eq-alfa)

Uma árvore (rede sem circuitos) tem $\mu = \alpha = 0$ e $\beta < 1$; uma triangulação planar maximal tem $\gamma = \alpha = 1$.

In [ ]:
G_u = ox.convert.to_undirected(G)
G_simples = nx.Graph(G_u)  # grafo simples: apenas a existência de ligação importa aqui
G_simples.remove_edges_from(nx.selfloop_edges(G_simples))

n_s, m_s = G_simples.number_of_nodes(), G_simples.number_of_edges()
p_s = nx.number_connected_components(G_simples)

indices = pd.Series({
    "n": n_s, "m": m_s, "componentes (p)": p_s,
    "beta = m/n": m_s / n_s,
    "mu = m - n + p": m_s - n_s + p_s,
    "gamma = m / 3(n-2)": m_s / (3 * (n_s - 2)),
    "alpha = (m-n+p) / (2n-5)": (m_s - n_s + p_s) / (2 * n_s - 5),
}).round(3)
indices.to_frame("valor")

:::{note}
Os limites $3(n-2)$ e $2n-5$ valem para grafos **planares**. Redes viárias são apenas aproximadamente planares, pois viadutos e túneis criam cruzamentos sem interseção, mas os índices continuam úteis para comparações relativas. Aqui a conversão para grafo simples com `nx.Graph` é adequada porque apenas a existência de ligação importa; para análises com pesos, lembre-se do aviso sobre arestas paralelas no {ref}`cap-fundamentos-grafos`.
:::

## Indicadores de conectividade

### Densidade da rede

A densidade {eq}`eq-densidade` compara o número de arcos com o máximo possível. Para o multidígrafo, a NetworkX usa $\rho = m / [n(n-1)]$:

In [ ]:
print(f"Densidade do multidígrafo: {nx.density(G):.5f}")
print(f"Densidade do grafo simples não direcionado: {nx.density(G_simples):.5f}")

Como discutido no {ref}`cap-fundamentos-grafos`, em redes espaciais $\rho \approx \bar{k}/n$ tende a zero à medida que $n$ cresce. A densidade **não é comparável** entre áreas de tamanhos diferentes e diz pouco sobre a qualidade da conectividade. Para comparações, prefira $\bar{k}$, os índices $\gamma$ e $\alpha$ ou as densidades por área.

### Assortatividade de grau

A **assortatividade de grau** $r$ {cite:p}`newman2002` é o coeficiente de correlação de Pearson entre os graus das extremidades das arestas:

$$
r = \frac{\sum_{(u,v) \in E} (k_u - \bar{k}_e)(k_v - \bar{k}_e)}{\sqrt{\sum_{(u,v)\in E} (k_u - \bar{k}_e)^2}\,\sqrt{\sum_{(u,v)\in E} (k_v - \bar{k}_e)^2}}, \qquad -1 \leq r \leq 1,
$$ (eq-assortatividade)

em que $\bar{k}_e$ é o grau médio das extremidades das arestas. Com $r > 0$, nós de grau alto tendem a se ligar entre si (rede **assortativa**); com $r < 0$, nós de grau alto tendem a se ligar a nós de grau baixo (rede **dissortativa**).

In [ ]:
r_multidigrafo = nx.degree_assortativity_coefficient(G)
r_simples = nx.degree_assortativity_coefficient(G_simples)
print(f"r (multidígrafo, grau total): {r_multidigrafo:.3f}")
print(f"r (grafo simples):            {r_simples:.3f}")

:::{warning}
Em redes espaciais, a faixa de graus é muito estreita (tipicamente de 1 a 5), e $r$ é sensível a detalhes de modelagem: direção, arestas paralelas, simplificação e consolidação de interseções. No multidígrafo, por exemplo, $r$ captura em parte a correlação entre **tipos de via**: segmentos de mão dupla ligam nós que têm, ambos, graus elevados. Por isso, $r$ pode ser positivo ou negativo dependendo da representação. Interprete a assortatividade de redes viárias com cautela e sempre informando o grafo utilizado.
:::

### Eficiência global e local

A **eficiência** {cite:p}`latora2001` mede quão facilmente a informação ou o fluxo circulam na rede, usando o inverso das distâncias, o que permite tratar pares desconectados ($1/\infty = 0$):

$$
E_{\text{glob}}(G) = \frac{1}{n(n-1)} \sum_{i \neq j} \frac{1}{d_{ij}}, \qquad
E_{\text{loc}}(G) = \frac{1}{n} \sum_{i \in V} E_{\text{glob}}\big(G[N(i)]\big),
$$ (eq-eficiencia)

em que $G[N(i)]$ é o subgrafo induzido pelos vizinhos de $i$. A eficiência local mede a tolerância a falhas: quão bem os vizinhos de um nó permanecem conectados se ele for removido.

In [ ]:
E_glob = nx.global_efficiency(G_simples)
E_loc = nx.local_efficiency(G_simples)
print(f"Eficiência global (topológica): {E_glob:.3f}")
print(f"Eficiência local média:         {E_loc:.3f}")

:::{important}
As funções `nx.global_efficiency` e `nx.local_efficiency` operam apenas sobre grafos **não direcionados** e usam distância **topológica** (número de arestas), ignorando comprimentos. Em redes espaciais, uma versão mais informativa normaliza a eficiência pela de uma rede ideal em que todos os pares estão ligados em linha reta {cite:p}`latora2001`:

$$
E_{\text{glob}}^{\text{esp}} = \frac{\sum_{i \neq j} 1/d_{ij}}{\sum_{i \neq j} 1/d^{E}_{ij}},
$$ (eq-eficiencia-espacial)

em que $d_{ij}$ é a distância na rede e $d^E_{ij}$ é a distância euclidiana. Essa razão está entre 0 e 1 e pode ser lida como o inverso de uma circuidade média entre pares (ver Exercício 7).
:::

A eficiência local de redes viárias costuma ser **baixa**. A razão é geométrica: em uma malha em grade, os quatro vizinhos de um cruzamento não são adjacentes entre si, de modo que $G[N(i)]$ não possui arestas. Isso reflete a ausência de triângulos, típica de malhas quase planares com quarteirões, e não uma "fragilidade" da rede.

## Centralidade

**Centralidade** é o termo genérico para medidas que atribuem a cada nó (ou aresta) um valor de importância estrutural. Não existe uma centralidade "correta": cada medida formaliza uma ideia diferente de importância {cite:p}`freeman1978`. Em redes viárias, as três medidas clássicas respondem a perguntas distintas:

- **grau**: quantas ligações diretas o nó possui? (conectividade **local**)
- **proximidade**: quão perto o nó está de todos os demais? (**acessibilidade**)
- **intermediação**: quantos caminhos mínimos passam pelo nó? (**potencial de passagem de fluxo**)

A escolha entre o multidígrafo $G$ e o grafo não direcionado $G_u$ é uma decisão de modelagem. Para pedestres, o sentido das vias é irrelevante. Para veículos, especialmente em áreas centrais, ele é determinante. Vejamos quanto a rede da Liberdade é marcada por vias de mão única:

In [ ]:
cores = ["tab:red" if d.get("oneway") else "tab:blue" for _, _, d in G.edges(data=True)]
fig, ax = ox.plot_graph(G, edge_color=cores, edge_linewidth=1, node_size=0, bgcolor="white",
                        show=False, close=False)
ax.set_title("Mão única (vermelho) × mão dupla (azul)")
plt.show()

prop_mao_unica = np.mean([bool(d.get("oneway")) for _, _, d in G.edges(data=True)])
print(f"Proporção de arcos em vias de mão única: {prop_mao_unica:.1%}")

:::{note}
Os grafos obtidos com a OSMnx são **multidígrafos** (`MultiDiGraph`), ou seja, **direcionados**. Em vias de mão dupla, existem dois arcos opostos. Qualquer análise que ignore o sentido deve converter o grafo explicitamente, de preferência com `ox.convert.to_undirected`.
:::

Como a proporção de vias de mão única é expressiva, calcularemos as centralidades no multidígrafo, distinguindo entrada e saída, e as compararemos com a versão não direcionada. Para facilitar a visualização, definimos uma função auxiliar que colore e dimensiona os nós segundo um atributo:

In [ ]:
def plot_centralidade(grafo, atributo, titulo, ax, escala=60, cmap="plasma"):
    """Plota a rede com nós coloridos e dimensionados pelo valor (normalizado) do atributo."""
    valores = pd.Series(nx.get_node_attributes(grafo, atributo))
    norm = (valores - valores.min()) / (valores.max() - valores.min() + 1e-12)
    cores = ox.plot.get_node_colors_by_attr(grafo, atributo, cmap=cmap)
    ox.plot_graph(grafo, ax=ax, node_color=cores, node_size=(5 + escala * norm).tolist(),
                  node_zorder=2, edge_color="#999999", edge_linewidth=0.5, bgcolor="white",
                  show=False, close=False)
    ax.set_title(titulo, fontsize=11)

### Centralidade de grau

A centralidade de grau normaliza o grau pelo máximo possível em um grafo simples:

$$
C_D(v) = \frac{\deg(v)}{n-1}, \qquad C_D^{-}(v) = \frac{\deg^{-}(v)}{n-1}, \qquad C_D^{+}(v) = \frac{\deg^{+}(v)}{n-1}.
$$ (eq-centralidade-grau)

Em multidígrafos, $\deg(v) = \deg^-(v) + \deg^+(v)$ e arestas paralelas são contadas, de modo que $C_D(v)$ pode exceder 1. Isso não ocorre na prática em redes viárias, mas explica por que os valores são tão pequenos: com $n$ na casa das centenas, $C_D(v)$ é da ordem de $10^{-2}$.

In [ ]:
C_D_in = nx.in_degree_centrality(G)
C_D_out = nx.out_degree_centrality(G)
C_D = nx.degree_centrality(G)

nx.set_node_attributes(G, C_D_in, "grau_entrada")
nx.set_node_attributes(G, C_D_out, "grau_saida")
nx.set_node_attributes(G, C_D, "grau_total")

pd.DataFrame({"entrada": C_D_in, "saída": C_D_out, "total": C_D}).describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (atr, tit) in zip(axes, [("grau_entrada", "Grau de entrada"),
                                 ("grau_saida", "Grau de saída"),
                                 ("grau_total", "Grau total")]):
    plot_centralidade(G, atr, tit, ax)
plt.tight_layout()
plt.show()

Como o grau assume poucos valores discretos, sua distribuição é mais bem apresentada como tabela de frequências do que como histograma contínuo:

In [ ]:
pd.DataFrame({
    "entrada": pd.Series(dict(G.in_degree())).value_counts(),
    "saída": pd.Series(dict(G.out_degree())).value_counts(),
    "total": pd.Series(dict(G.degree())).value_counts(),
}).fillna(0).astype(int).sort_index().rename_axis("grau")

Nós com $\deg^- \neq \deg^+$ revelam **assimetrias de circulação**: interseções em que mais vias chegam do que saem, ou o contrário. Esse padrão é típico de binários de mão única, pares de vias paralelas com sentidos opostos.

### Centralidade de proximidade

A centralidade de proximidade {cite:p}`sabidussi1966` é o inverso da distância média de um nó aos demais. Em um grafo direcionado, é preciso escolher o sentido das distâncias:

$$
C_C^{\text{in}}(u) = \frac{n-1}{\sum_{v \neq u} d_G(v, u)}, \qquad
C_C^{\text{out}}(u) = \frac{n-1}{\sum_{v \neq u} d_G(u, v)} .
$$ (eq-closeness)

- $C_C^{\text{in}}(u)$, a **proximidade de entrada**, mede quão facilmente $u$ **é alcançado** a partir dos demais nós, ou seja, sua acessibilidade como **destino**.
- $C_C^{\text{out}}(u)$, a **proximidade de saída**, mede quão facilmente se **parte** de $u$ para alcançar os demais, ou seja, sua acessibilidade como **origem**.

:::{warning}
Na NetworkX, `nx.closeness_centrality(G)` usa, para grafos direcionados, a **distância de entrada** $d_G(v, u)$. Portanto, ela calcula $C_C^{\text{in}}$. Para obter $C_C^{\text{out}}$, aplica-se a função ao grafo **invertido**, `G.reverse()`, no qual cada arco $(u,v)$ se torna $(v,u)$. Inverter essa correspondência é um erro frequente, que troca origens por destinos na interpretação.
:::

Quando nem todos os nós são mutuamente alcançáveis, o que é comum em recortes de redes viárias por causa de vias de mão única que "saem" da área de estudo, a NetworkX aplica por padrão a correção de Wasserman e Faust (`wf_improved=True`). A soma de distâncias é calculada apenas sobre os $r(u)$ nós alcançáveis, e o resultado é ponderado por $(r(u)-1)/(n-1)$, penalizando nós que alcançam (ou são alcançados por) poucos outros.

Usamos o comprimento das vias como peso (`distance="length"`). Os valores resultantes estão em $\text{m}^{-1}$; multiplicando por 1000, obtemos o inverso da distância média em quilômetros.

In [ ]:
C_C_in = nx.closeness_centrality(G, distance="length")                # distâncias até u
C_C_out = nx.closeness_centrality(G.reverse(), distance="length")     # distâncias a partir de u
C_C_u = nx.closeness_centrality(G_u, distance="length")               # sem sentido de circulação

nx.set_node_attributes(G, C_C_in, "proximidade_entrada")
nx.set_node_attributes(G, C_C_out, "proximidade_saida")
nx.set_node_attributes(G, C_C_u, "proximidade_nao_direcionada")

proximidade = pd.DataFrame({"entrada": C_C_in, "saída": C_C_out, "não direcionada": C_C_u})
(1 / proximidade).describe().round(0).rename(index={"mean": "média"}).T.add_suffix(" (m)")

A tabela acima apresenta o **inverso** da proximidade, que tem interpretação direta: a distância média, em metros, pela rede (corrigida pelo fator de alcance). A {numref}`fig-centralidade-proximidade` mostra os três campos de proximidade.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (atr, tit) in zip(axes, [("proximidade_entrada", "Proximidade de entrada"),
                                 ("proximidade_saida", "Proximidade de saída"),
                                 ("proximidade_nao_direcionada", "Proximidade (não direcionada)")]):
    plot_centralidade(G, atr, tit, ax)
plt.tight_layout()
plt.show()

Na versão não direcionada, a proximidade forma tipicamente um **gradiente centro–periferia**: é máxima no centro geométrico da área de estudo e decai em direção às bordas. Esse é o *efeito de borda*. Como a rede foi recortada pelo limite do bairro, nós próximos ao limite parecem periféricos porque suas vizinhanças reais, fora do recorte, foram excluídas. Nas versões direcionadas, o gradiente é perturbado pelo sistema de mão única: nós junto a vias que "entram" na área são bons destinos, mas maus pontos de partida, e vice-versa.

:::{tip}
Para atenuar o efeito de borda, baixe a rede com uma **área de amortecimento** (*buffer*) em torno da área de estudo, calcule as centralidades na rede ampliada e só então recorte os resultados para a área de interesse.
:::

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(proximidade["entrada"] * 1000, proximidade["saída"] * 1000, s=10, alpha=0.6)
lim = [proximidade[["entrada", "saída"]].min().min() * 1000, proximidade[["entrada", "saída"]].max().max() * 1000]
ax.plot(lim, lim, "k--", lw=1, label="entrada = saída")
ax.set_xlabel(r"$C_C^{in}$ (km$^{-1}$)")
ax.set_ylabel(r"$C_C^{out}$ (km$^{-1}$)")
ax.set_title("Assimetria entre acessibilidade de entrada e de saída")
ax.legend()
plt.show()

Pontos afastados da diagonal são nós com forte assimetria entre a acessibilidade como origem e como destino. Em uma rede sem vias de mão única, todos estariam sobre a diagonal.

### Centralidade de intermediação

A centralidade de intermediação {cite:p}`freeman1977` mede a fração de caminhos mínimos que passam por um nó:

$$
C_B(v) = \frac{1}{(n-1)(n-2)} \sum_{\substack{s \neq v \neq t}} \frac{\sigma_{st}(v)}{\sigma_{st}},
$$ (eq-betweenness)

em que $\sigma_{st}$ é o número de caminhos mínimos de $s$ a $t$ e $\sigma_{st}(v)$ é o número desses caminhos que passam por $v$. O fator de normalização é $1/[(n-1)(n-2)]$ para grafos direcionados e o dobro dele para grafos não direcionados, pois cada par é contado uma única vez. Diferentemente do grau e da proximidade, a intermediação é uma medida de **fluxo potencial**: sob a hipótese de que cada par origem–destino gera uma viagem pelo caminho mínimo, ela é proporcional ao volume que atravessaria cada nó.

O algoritmo de {cite:t}`brandes2001` calcula $C_B$ para todos os nós em tempo $O(nm + n^2 \log n)$ em grafos ponderados, o que é viável para bairros e, com amostragem (argumento `k`), para cidades inteiras.

In [ ]:
C_B = nx.betweenness_centrality(G, weight="length", normalized=True)
nx.set_node_attributes(G, C_B, "intermediacao")

pd.Series(C_B).describe().round(4).to_frame("intermediação")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1]})
plot_centralidade(G, "intermediacao", "Intermediação (nós)", axes[0], escala=120)
valores = pd.Series(C_B)
axes[1].hist(valores[valores > 0], bins=np.logspace(np.log10(valores[valores > 0].min()), np.log10(valores.max()), 30),
             color="tab:orange", edgecolor="black")
axes[1].set_xscale("log")
axes[1].set_xlabel(r"$C_B(v)$ (escala log)")
axes[1].set_ylabel("Número de nós")
axes[1].set_title("Distribuição da intermediação")
plt.tight_layout()
plt.show()

A distribuição da intermediação em redes viárias é fortemente **assimétrica**: a maioria dos nós tem valores próximos de zero, e poucos nós, alinhados ao longo dos eixos que atravessam a área, concentram a maior parte dos caminhos mínimos. Essa estrutura hierárquica emerge da topologia, mesmo sem nenhuma informação sobre classe viária ou demanda real, e costuma coincidir com as vias arteriais {cite:p}`porta2006`.

#### Intermediação de arestas

Em redes viárias, a unidade de interesse costuma ser o **segmento de rua**, e não a interseção. A intermediação de arestas é definida de forma análoga a {eq}`eq-betweenness`, contando os caminhos mínimos que atravessam cada aresta:

In [ ]:
C_B_e = nx.edge_betweenness_centrality(G, weight="length", normalized=True)
nx.set_edge_attributes(G, C_B_e, "intermediacao")

cores_arestas = ox.plot.get_edge_colors_by_attr(G, "intermediacao", cmap="inferno")
valores_e = pd.Series(C_B_e)
larguras = (0.3 + 4 * (valores_e / valores_e.max())).reindex(list(G.edges(keys=True))).tolist()
fig, ax = ox.plot_graph(G, node_size=0, edge_color=cores_arestas, edge_linewidth=larguras, bgcolor="black")

:::{note}
Para multigrafos, a NetworkX indexa `edge_betweenness_centrality` pelas triplas $(u, v, k)$, compatíveis com `nx.set_edge_attributes` e com a ordem de `G.edges(keys=True)` usada pela OSMnx.
:::

### Comparação entre centralidades

As três medidas capturam aspectos diferentes da importância estrutural. A correlação de postos de Spearman ($\rho_s$) quantifica o quanto elas concordam no **ordenamento** dos nós:

In [ ]:
centralidades = pd.DataFrame({
    "grau": C_D,
    "proximidade (in)": C_C_in,
    "proximidade (out)": C_C_out,
    "proximidade (não dir.)": C_C_u,
    "intermediação": C_B,
})
centralidades.corr(method="spearman").round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, (atr, tit) in zip(axes, [("grau_total", "Grau"),
                                 ("proximidade_nao_direcionada", "Proximidade"),
                                 ("intermediacao", "Intermediação")]):
    plot_centralidade(G, atr, tit, ax)
plt.tight_layout()
plt.show()

Em geral, observa-se que:

- o **grau** é uma medida local e pouco variável, que distingue sobretudo becos sem saída, junções em "T" e cruzamentos;
- a **proximidade** varia suavemente no espaço e é dominada pela posição geométrica em relação ao recorte (efeito de borda);
- a **intermediação** é a mais seletiva e revela os corredores que estruturam a circulação.

A baixa correlação entre grau e intermediação é característica de redes espaciais: um nó de grau 3 pode ter intermediação altíssima se estiver sobre o único eixo que conecta duas regiões {cite:p}`barthelemy2011`.

## Orientação da rede

A **orientação** (*bearing*) de um segmento é o ângulo, medido no sentido horário a partir do norte geográfico, da reta que liga suas extremidades. Para extremidades com latitudes $\varphi_1, \varphi_2$ e diferença de longitude $\Delta\lambda$:

$$
\theta = \operatorname{atan2}\!\Big(\sin\Delta\lambda \cos\varphi_2,\; \cos\varphi_1\sin\varphi_2 - \sin\varphi_1\cos\varphi_2\cos\Delta\lambda\Big) \pmod{360^\circ}.
$$ (eq-bearing)

Como uma rua pode ser percorrida nos dois sentidos, a análise usa o grafo não direcionado e considera cada segmento com as orientações $\theta$ e $\theta + 180^\circ$. A OSMnx agrupa as orientações em $b$ classes (36 por padrão, de $10^\circ$ cada) e calcula a **entropia de orientação** de Shannon {cite:p}`boeing2019`:

$$
H_o = -\sum_{i=1}^{b} P(o_i) \ln P(o_i),
$$ (eq-entropia-orientacao)

em que $P(o_i)$ é a proporção de segmentos (ponderada, opcionalmente, pelo comprimento) na classe $i$. O máximo, $H_{\max} = \ln 36 \approx 3{,}584$, corresponde a orientações uniformemente distribuídas. Para uma grade perfeita, com apenas quatro direções, $H_g = \ln 4 \approx 1{,}386$. {cite:t}`boeing2019` propõe normalizar a entropia no **índice de ordem de orientação**:

$$
\varphi = 1 - \left(\frac{H_o - H_g}{H_{\max} - H_g}\right)^{2},
$$ (eq-ordem-orientacao)

que vale 1 para uma grade perfeita e se aproxima de 0 para orientações desordenadas.

In [ ]:
G_u = ox.add_edge_bearings(ox.convert.to_undirected(G))

H_o = ox.bearing.orientation_entropy(G_u, num_bins=36, weight="length")
H_max, H_g = math.log(36), math.log(4)
phi = 1 - ((H_o - H_g) / (H_max - H_g)) ** 2
print(f"Entropia de orientação H_o = {H_o:.3f} (máx. {H_max:.3f}; grade {H_g:.3f})")
print(f"Índice de ordem φ = {phi:.3f}")

In [ ]:
fig, ax = ox.plot_orientation(G_u, num_bins=36, weight="length", title="Liberdade, São Paulo", area=True)
plt.show()

A leitura do histograma polar deve ser feita em conjunto com $\varphi$. Picos bem definidos a cada $90^\circ$ e $\varphi$ elevado indicam uma malha ortogonal planejada. Picos múltiplos ou difusos e $\varphi$ baixo indicam um tecido orgânico ou condicionado pelo relevo, como é comum nas áreas centrais de cidades brasileiras que cresceram sobre colinas e fundos de vale. A orientação também tem implicações para a orientabilidade de pedestres e para a exposição solar das fachadas.

:::{tip}
Comparar $\varphi$ entre bairros ou cidades é uma forma compacta de caracterizar a morfologia urbana. Os resultados podem ser relacionados ao período de urbanização, ao relevo e ao tipo de parcelamento do solo {cite:p}`boeing2019`.
:::

## Síntese do capítulo

- `ox.basic_stats` fornece estatísticas topológicas, geométricas e combinadas; densidades exigem a área em m² como escalar, e `k_avg` difere de `streets_per_node_avg` por causa da duplicação de arcos.
- Os índices $\beta$, $\gamma$, $\alpha$ e $\mu$ resumem a conectividade de circuitos em redes aproximadamente planares.
- Densidade, assortatividade e eficiência topológica têm interpretação limitada em redes espaciais; sempre informe a representação usada (direcionada ou não, simples ou multigrafo, simplificada ou não).
- As centralidades de **grau**, **proximidade** e **intermediação** formalizam noções distintas de importância. No multidígrafo, `nx.closeness_centrality(G)` fornece a proximidade de **entrada**, e `G.reverse()` fornece a de **saída**.
- A intermediação revela a hierarquia funcional da malha e pode ser calculada para nós e para arestas.
- A **entropia de orientação** e o índice $\varphi$ quantificam o grau de ordem geométrica do traçado viário.

## Exercícios Práticos e Desafios

### Exercícios conceituais

1. Prove que, para qualquer grafo, $\bar{k} = 2\beta$. Por que, então, a OSMnx reporta `k_avg` e não $\beta$?
2. Uma árvore com $n$ nós tem quantas arestas? Calcule $\beta$, $\mu$, $\gamma$ e $\alpha$ para uma árvore e para uma grade retangular de $4 \times 4$ nós.
3. Explique, com um exemplo de três nós e dois arcos, por que `nx.closeness_centrality(G)` mede a acessibilidade como **destino** em grafos direcionados.
4. Por que a eficiência local de uma grade perfeita é zero? Isso significa que a grade é pouco robusta a falhas? Justifique.

### Exercícios computacionais

5. Refaça a análise de estatísticas básicas para um bairro planejado (por exemplo, o Plano Piloto de Brasília ou um loteamento em grade da sua cidade) e compare `streets_per_node_proportions`, `circuity_avg`, $\gamma$ e $\varphi$ com os da Liberdade.
6. Calcule a centralidade de intermediação **não direcionada** (sobre `G_u`) e compare-a com a direcionada por meio do coeficiente de Spearman. Quais nós mudam mais de posição no ordenamento? Onde estão localizados?
7. Implemente a eficiência espacial {eq}`eq-eficiencia-espacial` usando `nx.all_pairs_dijkstra_path_length` e a distância euclidiana no grafo projetado. Compare o valor obtido com o inverso de uma circuidade média entre pares.
8. Baixe a rede com um *buffer* de 500 m em torno da Liberdade (dica: projete o polígono do bairro, aplique `buffer(500)` e use `ox.graph_from_polygon` após reprojetar para WGS 84). Recalcule a proximidade, recorte os resultados para o bairro e quantifique o efeito de borda.

### Desafios

9. **Intermediação ponderada por tempo.** Após estudar o {ref}`cap-roteamento`, adicione velocidades e tempos de viagem às arestas (`ox.add_edge_speeds` e `ox.add_edge_travel_times`) e recalcule a intermediação com `weight="travel_time"`. Como a hierarquia de corredores muda quando o critério passa de distância para tempo?
10. **Vulnerabilidade.** Remova, um a um, os dez nós de maior intermediação e, a cada remoção, recalcule o tamanho da maior componente fortemente conexa e a eficiência global. Compare com a remoção de dez nós aleatórios (repita 30 vezes). O que o resultado indica sobre a robustez da rede a falhas direcionadas?